# Junction ore block clustering and diversity

In [2]:
# set working directory to project folder
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import plotly.io as pio
pio.renderers.default = "browser" 

import altair as alt
from itertools import combinations
import numpy as np
import math
import pypangraph as pp
from Bio import Phylo, SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

import random
import plotly.graph_objects as go
import plotly.express as px

import scipy.cluster.hierarchy as sch
from collections import defaultdict, deque, Counter
from Bio import Phylo

from pathlib import Path
import subprocess

from Bio.Phylo.TreeConstruction import DistanceCalculator
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

from junction_analysis.helpers import get_tree_order, cluster_by_tree, strip_newick_suffixes, write_shared_nodes_fasta, simplify_cluster_keys
from junction_analysis.junction_trees import build_tree_from_block_list, compute_pairwise_distances, cluster_tree_by_branch_length
import junction_analysis.pangraph_utils as pu
from junction_analysis.plotting import plot_junction_pangraph_interactive, plot_block_distance_distribution, plot_pairwise_distance_hist
from junction_analysis.consensus import find_consensus_paths, make_deduplicated_paths, filter_cluster_paths_by_block_freq, compute_cluster_consensus_paths, consensus_paths_and_assignments, find_consensus_paths_core
from junction_analysis.block_alignment import create_block_msas, summarize_block_msas, analyze_alignment, cluster_alignment, retrieve_cluster_assignments
from junction_analysis.annotate_insertions import get_insertions_deletions_from_consensus, write_insertions_fasta, summarize_deletions_consensus, load_all_deletions_summaries

In [3]:
_fname = f"../config/junction_stats.csv"
jdf = pd.read_csv(_fname)
jdf

,edge,n_iso,n_blocks,has_dupl,n_categories,majority_category,singleton,cat_entropy,n_nodes,min_length,max_length,mean_length,n_all_cores,core_left_length,core_right_length,transitive,nonempty_acc_len,nonempty_freq,pangenome_len
0,HUTOPWFGVH_f__WJBYSSJHSE_f,222,5,False,3,218,False,0.100354,11,18177,18954,18190.108108,218,16280.0,1699.0,False,727.5,0.018018,1684.0
1,IZQZNHJQRQ_f__ZVPGGEJIIF_f,222,3,False,2,221,True,0.028831,5,11740,12517,11743.500000,221,1530.0,10210.0,False,777.0,0.004505,768.0
2,JNLRGIQXOF_f__YQUHGDANHE_f,222,3,False,2,221,True,0.028831,5,28035,29472,28041.472973,221,23758.0,4277.0,False,1437.0,0.004505,1433.0
3,JMOMDSHCBS_r__SPDPCYMYDN_r,222,3,False,2,220,False,0.051397,5,7366,8143,7373.000000,220,2781.0,4585.0,False,777.0,0.009009,768.0
4,JKRDVEYDGL_f__OTPJRRWJRK_f,222,3,False,2,221,True,0.028831,5,10058,48233,10229.959459,221,7295.0,2763.0,False,38175.0,0.004505,37225.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,GPXHVRRZLC_f__KYQOKYBCOW_f,2,12,False,2,1,True,0.693147,18,65788,105315,85551.500000,0,18606.0,29024.0,False,34255.5,1.000000,100857.0
544,EOBHADSLFU_f__SKPHAXSFLS_r,2,1,False,1,2,False,0.000000,1,7231,7231,7231.000000,2,7231.0,7231.0,True,NaN,0.000000,3298.0
545,XXIWNZXZTK_f__ZLAJFQLBFQ_r,2,3,False,2,1,True,0.693147,6,83630,83630,83630.000000,2,31644.0,48921.0,False,NaN,0.000000,73408.0
546,YUOECYBHUS_f__ZTHKZYHPIX_f,2,20,True,2,1,True,0.693147,35,66710,186048,126379.000000,0,60232.0,5710.0,False,60437.0,1.000000,160404.0


In [5]:
clustering_results = {}

for junction_id in jdf["edge"].tolist():
    cluster_map_core, consensus_paths_core, path_dict, consensus_paths_plotting, assignment_df_plotting = find_consensus_paths_core(junction_id, plot_consensus=False, plot_pair_dist=False, clustering_bl_thresh=0.005, block_freq_thresh=0.7)
    clustering_results[junction_id] = cluster_map_core

Wrote FASTA: ../results/consensus_analysis/HUTOPWFGVH_f__WJBYSSJHSE_f/core_blocks.fa
Wrote alignment: ../results/consensus_analysis/HUTOPWFGVH_f__WJBYSSJHSE_f/core_blocks_aln.fa


FastTree Version 2.2.0 Double precision
Alignment: ../results/consensus_analysis/HUTOPWFGVH_f__WJBYSSJHSE_f/core_blocks_aln.fa
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Generalized Time-Reversible, CAT approximation with 20 rate categories
Initial topology in 0.04 seconds
Refining topology: 19 rounds ME-NNIs, 2 rounds ME-SPRs, 9 rounds ML-NNIs
      0.24 seconds: ME NNI round 7 of 19, 1 of 24 splits
      0.42 seconds: ME NNI round 13 of 19, 1 of 24 splits
Total branch-length 0.003 after 0.44 sec

WARNING! This alignment consists of closely-related and very-long sequences.
WARNING! FastTree (or other standard maximum-likelihood tools)
may not be appropriate for aligments of very closely-related sequences
like this one, as FastTree does not account for recombination or gene conversion

ML-NNI round 1: LogLk = -25715.767 NNIs 13 max delta

Wrote tree: ../results/consensus_analysis/HUTOPWFGVH_f__WJBYSSJHSE_f/core_blocks_aln.newick
Cluster 0: 222 / 222 isolates share the majority path
Wrote FASTA: ../results/consensus_analysis/IZQZNHJQRQ_f__ZVPGGEJIIF_f/core_blocks.fa
Wrote alignment: ../results/consensus_analysis/IZQZNHJQRQ_f__ZVPGGEJIIF_f/core_blocks_aln.fa


FastTree Version 2.2.0 Double precision
Alignment: ../results/consensus_analysis/IZQZNHJQRQ_f__ZVPGGEJIIF_f/core_blocks_aln.fa
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Generalized Time-Reversible, CAT approximation with 20 rate categories
Initial topology in 0.02 seconds
Refining topology: 17 rounds ME-NNIs, 2 rounds ME-SPRs, 8 rounds ML-NNIs
      0.10 seconds: ME NNI round 6 of 17, 1 of 16 splits
Total branch-length 0.002 after 0.18 sec

WARNING! This alignment consists of closely-related and very-long sequences.
WARNING! FastTree (or other standard maximum-likelihood tools)
may not be appropriate for aligments of very closely-related sequences
like this one, as FastTree does not account for recombination or gene conversion

      0.20 seconds: ML NNI round 1 of 8, 1 of 16 splits
ML-NNI round 1: LogLk = -16475.474 NNIs 12 max delta 0

Wrote tree: ../results/consensus_analysis/IZQZNHJQRQ_f__ZVPGGEJIIF_f/core_blocks_aln.newick
Cluster 0: 222 / 222 isolates share the majority path
Wrote FASTA: ../results/consensus_analysis/JNLRGIQXOF_f__YQUHGDANHE_f/core_blocks.fa
Wrote alignment: ../results/consensus_analysis/JNLRGIQXOF_f__YQUHGDANHE_f/core_blocks_aln.fa


...

Wrote tree: ../results/consensus_analysis/JVNRLCFAVD_f__PLTCZQCVRD_r/core_blocks_aln.newick
Cluster 0: 152 / 166 isolates share the majority path
Cluster 1: 48 / 52 isolates share the majority path
Skipping core: FASTA, alignment and tree already exist in ../results/consensus_analysis/RYYAQMEJGY_r__ZTHKZYHPIX_f
Cluster 0: 93 / 145 isolates share the majority path
Cluster 1: 1 / 1 (single isolate)
Cluster 2: 15 / 17 isolates share the majority path
Cluster 3: 1 / 1 (single isolate)
Cluster 4: 38 / 52 isolates share the majority path
Wrote FASTA: ../results/consensus_analysis/KGJXRPELGX_r__KIVDDRSTJR_r/core_blocks.fa
Wrote alignment: ../results/consensus_analysis/KGJXRPELGX_r__KIVDDRSTJR_r/core_blocks_aln.fa


FastTree Version 2.2.0 Double precision
Alignment: ../results/consensus_analysis/KGJXRPELGX_r__KIVDDRSTJR_r/core_blocks_aln.fa
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Generalized Time-Reversible, CAT approximation with 20 rate categories
Initial topology in 0.03 seconds
Refining topology: 18 rounds ME-NNIs, 2 rounds ME-SPRs, 9 rounds ML-NNIs
      0.16 seconds: ME NNI round 7 of 18, 1 of 21 splits
      0.34 seconds: ME NNI round 13 of 18, 1 of 21 splits
Total branch-length 0.001 after 0.35 sec

WARNING! This alignment consists of closely-related and very-long sequences.
WARNING! FastTree (or other standard maximum-likelihood tools)
may not be appropriate for aligments of very closely-related sequences
like this one, as FastTree does not account for recombination or gene conversion

ML-NNI round 1: LogLk = -20506.762 NNIs 9 max delta 

Wrote tree: ../results/consensus_analysis/KGJXRPELGX_r__KIVDDRSTJR_r/core_blocks_aln.newick
Cluster 0: 207 / 215 isolates share the majority path


Total time: 2.66 seconds Unique: 23/215 Bad splits: 0/20


ValueError: last_non_dup must not be None when encountering a duplicated node

In [ ]:
len(np.unique(list(cluster_map_core.values())))

In [ ]:
# TODO: how do I measure junction diversity
# average pairwise block jaccard distance?
# maybe compare to cat_entropy of each junction